# 12 Aggregations

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Compute several named aggregates at once with <code>agg</code>, use the full toolbox (<code>count</code>, <code>countDistinct</code>, <code>approx_count_distinct</code>, <code>sum</code>, <code>avg</code>, <code>collect_list</code>, <code>percentile</code>), and build multi-level summaries with <code>rollup</code>, <code>cube</code> and <code>pivot</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Gold-layer tables, dashboards and KPIs are aggregations. You need to produce several metrics in one pass, label them clearly, and know which functions are expensive (exact distinct counts, collecting lists) and which are cheap. Interviewers love <code>rollup</code> vs <code>cube</code>, <code>countDistinct</code> vs <code>approx_count_distinct</code>, and pivoting. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 agg() and aggregate functions</b><br>
<code>df.groupBy(keys).agg(f1.alias("a"), f2.alias("b"), ...)</code> computes many aggregates in <b>one</b> pass over the data and <b>one</b> shuffle.
<ul><li><b>Basic:</b> <code>count</code>, <code>sum</code>, <code>avg</code>/<code>mean</code>, <code>min</code>, <code>max</code></li><li><b>Distinct:</b> <code>countDistinct</code> (exact, expensive), <code>approx_count_distinct</code> (HyperLogLog, cheap, ~2-5% error)</li><li><b>Collections:</b> <code>collect_list</code> (keeps duplicates), <code>collect_set</code> (unique)</li><li><b>Statistics:</b> <code>stddev</code>, <code>variance</code>, <code>percentile_approx</code>, <code>median</code></li><li><b>Value at extreme:</b> <code>max_by(value, ordering)</code>, <code>min_by(...)</code></li><li><b>Multi-level:</b> <code>rollup</code> (hierarchical subtotals), <code>cube</code> (all combinations), <code>pivot</code> (values become columns)</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retail chain's Gold table <code>daily_store_kpis</code> has, per store and day: revenue, number of orders, unique customers, average basket size, the best-selling product and the 95th percentile order value. It is computed with a single <code>groupBy("store_id", "date").agg(...)</code>. The finance team also wants subtotals per region and a grand total, which is exactly what <code>rollup</code> gives.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.groupBy("store").agg(
    F.sum("amount").alias("revenue"),
    F.count("*").alias("orders"),
    F.countDistinct("customer_id").alias("customers"),
    F.round(F.avg("amount"), 2).alias("avg_order"),
)

df.agg(F.max("amount"))                              # whole DataFrame, no grouping
df.rollup("region", "store").agg(F.sum("amount"))    # subtotals + grand total
df.cube("region", "channel").agg(F.sum("amount"))    # every combination
df.groupBy("store").pivot("channel", ["web", "shop"]).sum("amount")
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a sales DataFrame with regions, stores, channels, customers and products.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 12-row table with <code>region</code>, <code>store</code>, <code>channel</code>, <code>customer_id</code>, <code>product</code>, <code>amount</code>.
</div>

In [0]:
from pyspark.sql import functions as F

sales = spark.createDataFrame(
    [
        ("South", "Chennai", "shop", "C1", "Laptop", 1200.0),
        ("South", "Chennai", "web", "C2", "Mouse", 25.0),
        ("South", "Chennai", "web", "C1", "Keyboard", 45.0),
        ("South", "Bangalore", "shop", "C3", "Laptop", 1100.0),
        ("South", "Bangalore", "web", "C4", "Monitor", 300.0),
        ("North", "Delhi", "shop", "C5", "Phone", 700.0),
        ("North", "Delhi", "shop", "C5", "Case", 20.0),
        ("North", "Delhi", "web", "C6", "Phone", 680.0),
        ("North", "Jaipur", "shop", "C7", "Mouse", 22.0),
        ("North", "Jaipur", "web", "C8", "Laptop", 1150.0),
        ("North", "Jaipur", "web", "C7", "Monitor", 310.0),
        ("South", "Chennai", "shop", "C9", "Phone", 720.0),
    ],
    "region STRING, store STRING, channel STRING, customer_id STRING, product STRING, amount DOUBLE",
)
display(sales)

## 1. Several named aggregates with `agg`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes six store KPIs in one <code>agg</code> call.<br><br>
<b>Why it matters</b><br>One <code>agg</code> means one scan and one shuffle, no matter how many metrics. Always <code>alias</code> them so the output schema is clean.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows, one per store. Chennai: revenue 1990.0, 4 orders, 3 customers, average 497.5, biggest order 1200.0.
</div>

In [0]:
store_kpis = sales.groupBy("store").agg(
    F.sum("amount").alias("revenue"),
    F.count("*").alias("orders"),
    F.countDistinct("customer_id").alias("customers"),
    F.round(F.avg("amount"), 2).alias("avg_order"),
    F.max("amount").alias("biggest_order"),
    F.max_by("product", "amount").alias("top_product"),
)
store_kpis.orderBy("store").show()

## 2. Aggregating the whole DataFrame

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>df.agg(...)</code> without a <code>groupBy</code> for grand totals, and the dictionary shortcut.<br><br>
<b>Why it matters</b><br>Quick totals and data quality checks (row counts, min and max dates) use this form.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row with total revenue 6272.0, 12 orders, min 20.0 and max 1200.0. Then the same max and min via the dictionary form.
</div>

In [0]:
sales.agg(
    F.sum("amount").alias("total_revenue"),
    F.count("*").alias("orders"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
).show()

# Dictionary form: quick, but you can't alias and you get one function per column.
sales.agg({"amount": "max", "customer_id": "count"}).show()

## 3. Exact vs approximate distinct counts

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares <code>countDistinct</code> with <code>approx_count_distinct</code> on 1 million generated rows.<br><br>
<b>Why it matters</b><br>Exact distinct counts need every distinct value to be shuffled and kept in memory, so they are expensive on big data. <code>approx_count_distinct</code> uses the HyperLogLog++ sketch, with a small fixed memory per group and a typical error of a few percent. Dashboards usually accept the approximation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The exact count is 100,000. The approximate counts are close to it (within a few percent).
</div>

In [0]:
big = spark.range(1_000_000).withColumn("user_id", F.col("id") % 100_000)

big.agg(
    F.countDistinct("user_id").alias("exact"),
    F.approx_count_distinct("user_id").alias("approx_default"),       # rsd = 0.05
    F.approx_count_distinct("user_id", 0.01).alias("approx_1pct"),    # more accurate, more memory
).show()

## 4. Collecting values: `collect_list` and `collect_set`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Gathers each customer's products into a list (with duplicates) and a set (unique).<br><br>
<b>Why it matters</b><br>Useful for building nested outputs or features ("products bought"). But collecting puts every value of a group into one row in memory, so it can cause out-of-memory errors on large groups.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
The <b>order</b> of elements in <code>collect_list</code> is not guaranteed. Sort the array with <code>array_sort</code> if order matters.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C5 has <code>[Case, Phone]</code> (sorted), and C1 has <code>[Keyboard, Laptop]</code>. The <code>n_products</code> column counts the set's size.
</div>

In [0]:
sales.groupBy("customer_id").agg(
    F.array_sort(F.collect_list("product")).alias("products"),
    F.size(F.collect_set("channel")).alias("n_channels"),
    F.size(F.collect_set("product")).alias("n_products"),
).orderBy("customer_id").show(truncate=False)

## 5. Statistics and percentiles

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes standard deviation, median and percentiles of order value per region.<br><br>
<b>Why it matters</b><br>Averages hide outliers. The median and the 95th percentile describe typical and high-end orders much better. <code>percentile_approx</code> is the scalable version.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per region with <code>stddev</code>, <code>median</code>, and an array <code>[p50, p90]</code> from <code>percentile_approx</code>.
</div>

In [0]:
sales.groupBy("region").agg(
    F.round(F.stddev("amount"), 1).alias("stddev"),
    F.median("amount").alias("median"),
    F.percentile_approx("amount", [0.5, 0.9]).alias("p50_p90"),
).orderBy("region").show(truncate=False)

## 6. Conditional aggregates and ratios

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Splits revenue by channel and computes the web share per region, all in one aggregation.<br><br>
<b>Why it matters</b><br>This "pivot by hand" with <code>when</code> gives full control over column names and works well with ratios.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
North: shop 742.0, web 2140.0, web share about 74%. South: shop 3020.0, web 370.0, web share about 11%.
</div>

In [0]:
by_channel = sales.groupBy("region").agg(
    F.sum(F.when(F.col("channel") == "shop", F.col("amount")).otherwise(0)).alias("shop_revenue"),
    F.sum(F.when(F.col("channel") == "web", F.col("amount")).otherwise(0)).alias("web_revenue"),
    F.sum("amount").alias("total"),
)
by_channel.withColumn("web_share_pct", F.round(F.col("web_revenue") / F.col("total") * 100, 1)) \
    .orderBy("region").show()

## 7. pivot: values become columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Pivots <code>channel</code> into columns, first letting Spark find the values and then passing them explicitly.<br><br>
<b>Why it matters</b><br>Pivoting produces the wide tables that business users expect in spreadsheets. Passing the list of values avoids an extra job to compute distinct values, and protects you from unexpected new values creating new columns.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two identical tables: one row per store with <code>shop</code> and <code>web</code> revenue columns (Chennai: 1920.0 and 70.0). Every store sells in both channels here. If one didn't, the first table would show null and the second 0, thanks to <code>fillna</code>.
</div>

In [0]:
sales.groupBy("store").pivot("channel").sum("amount").orderBy("store").show()

# Explicit values: no extra job to discover them, and a fixed output schema.
(
    sales.groupBy("store")
    .pivot("channel", ["shop", "web"])
    .agg(F.sum("amount"))
    .fillna(0)
    .orderBy("store")
    .show()
)

## 8. rollup and cube: subtotals

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds hierarchical subtotals with <code>rollup</code>, then every combination with <code>cube</code>, and labels subtotal rows with <code>grouping()</code>.<br><br>
<b>Why it matters</b><br><code>rollup("region", "store")</code> gives store totals, region subtotals and a grand total in one query, which is exactly what finance reports need. <code>cube</code> adds totals for every combination of the columns. In the output, null in a key column means "all values", and <code>grouping()</code> tells real nulls apart from subtotal rows.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>rollup</code>: 4 store rows + 2 region subtotals + 1 grand total = 7 rows. <code>cube</code> on region and channel: 4 combinations + 2 region totals + 2 channel totals + 1 grand total = 9 rows.
</div>

In [0]:
(
    sales.rollup("region", "store")
    .agg(F.sum("amount").alias("revenue"), F.grouping_id().alias("level"))
    .withColumn("region", F.coalesce("region", F.lit("ALL REGIONS")))
    .withColumn("store", F.when(F.col("level") > 0, F.lit("ALL STORES")).otherwise(F.col("store")))
    .orderBy("level", "region", "store")
    .show()
)

sales.cube("region", "channel").agg(F.sum("amount").alias("revenue")) \
    .orderBy(F.col("region").asc_nulls_last(), F.col("channel").asc_nulls_last()).show()

## 9. The same in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a KPI query plus <code>GROUP BY ROLLUP</code> in SQL.<br><br>
<b>Why it matters</b><br>Databricks SQL dashboards and many exam questions use SQL syntax.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Region-level KPIs, then the rollup with subtotals.
</div>

In [0]:
sales.createOrReplaceTempView("sales_v")

spark.sql("""
    SELECT region,
           sum(amount)                         AS revenue,
           count(DISTINCT customer_id)         AS customers,
           round(avg(amount), 2)               AS avg_order,
           count_if(channel = 'web')           AS web_orders
    FROM sales_v
    GROUP BY region
    ORDER BY region
""").show()

spark.sql("""
    SELECT region, store, sum(amount) AS revenue
    FROM sales_v
    GROUP BY ROLLUP (region, store)
    ORDER BY region NULLS LAST, store NULLS LAST
""").show()

## Under the hood

```
groupBy("store").agg(sum, count, avg, ...)
  HashAggregate(partial_sum, partial_count, partial_avg ...)    <- per partition
  Exchange hashpartitioning(store)                              <- ONE shuffle for all metrics
  HashAggregate(sum, count, avg ...)                            <- final

countDistinct("customer_id")
  needs an extra aggregation level: first dedupe (store, customer_id), then count
  -> more shuffle data, more memory

approx_count_distinct("customer_id")
  each partition builds a small HyperLogLog sketch, sketches are merged after the shuffle
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> every grouped aggregation shuffles once, no matter how many metrics you compute.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `agg`, `rollup`, `cube` and `pivot` are lazy. Without explicit values, `pivot` runs an **extra job** immediately to find the distinct values.

**Interview point:** `avg` is computed as a partial `(sum, count)` per partition, then `total_sum / total_count`. You can't average partial averages, which is why the partial step stores both numbers. Look for `partial_avg` and the extra aggregate levels for `count(distinct)` in the plans below.

In [0]:
gen = spark.range(1_000_000).withColumn("store", F.col("id") % 10).withColumn("cust", F.col("id") % 1000)

print("---- avg + sum ----")
gen.groupBy("store").agg(F.avg("id"), F.sum("id")).explain()

print("---- countDistinct ----")
gen.groupBy("store").agg(F.countDistinct("cust")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: ugly column names like <code>sum(amount)</code></b><br>
Use <code>agg(...)</code> with <code>.alias(...)</code> instead of the shortcut methods.<br><br>
**⛔ Problem: <code>countDistinct</code> is very slow or runs out of memory**<br>Exact distinct counts are expensive. Use <code>approx_count_distinct</code> when a small error is fine.<br><br>
**⛔ Problem: executor out of memory with <code>collect_list</code>**<br>A huge group is collected into one row. Aggregate differently, or limit the list with <code>slice</code>.<br><br>
**⛔ Problem: <code>pivot</code> triggers an extra Spark job**<br>Spark must discover the distinct values. Pass them explicitly: <code>pivot("col", ["a", "b"])</code>.<br><br>
<b>⛔ Problem: "average of averages" is wrong</b><br>Averaging per-group averages ignores group sizes. Recompute from sums and counts.<br><br>
<b>⛔ Problem: subtotal rows look like null data</b><br>In <code>rollup</code> and <code>cube</code> output, null keys mean "all". Use <code>grouping()</code> or <code>grouping_id()</code> to label them.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you compute several aggregates in one go, and why is it better than separate queries?</b><br>
<code>groupBy(...).agg(f1.alias(...), f2.alias(...))</code>. Spark computes all of them in one scan and one shuffle. Separate queries would scan and shuffle the data again for each metric.<br><br>

<b>🎤 2. What is the difference between <code>countDistinct</code> and <code>approx_count_distinct</code>?</b><br>
<code>countDistinct</code> is exact, but it must shuffle and track every distinct value, which is expensive at scale. <code>approx_count_distinct</code> uses HyperLogLog++ sketches with fixed memory and a configurable relative error, about 5% by default. It is much cheaper.<br><br>

<b>🎤 3. What is the difference between <code>rollup</code> and <code>cube</code>?</b><br>
<code>rollup(a, b)</code> produces hierarchical subtotals: (a, b), (a) and the grand total. <code>cube(a, b)</code> produces every combination: (a, b), (a), (b) and the grand total. Cube grows as 2 to the power of the number of columns.<br><br>

<b>🎤 4. How do you tell a subtotal row from a real null in rollup output?</b><br>
Use <code>grouping(col)</code>, which is 1 when the column is aggregated away, or <code>grouping_id()</code> for a bitmask across all grouping columns.<br><br>

<b>🎤 5. What does <code>pivot</code> do, and how do you make it efficient?</b><br>
It turns distinct values of a column into separate columns with an aggregate in each cell. Passing the list of values explicitly avoids an extra job to compute them and fixes the output schema.<br><br>

<b>🎤 6. How does Spark compute <code>avg</code> in a distributed way?</b><br>
Each partition computes a partial sum and count. After the shuffle, they are combined into the total sum and total count, and the average is their ratio. Averaging partial averages would be wrong.<br><br>

<b>🎤 7. What is the difference between <code>collect_list</code> and <code>collect_set</code>?</b><br>
<code>collect_list</code> keeps all values including duplicates. <code>collect_set</code> keeps only unique values. Neither guarantees order, and both can use a lot of memory on large groups.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A dashboard needs the number of unique visitors per day on a 5 billion row table, and a 2% error is acceptable. Which function is the best choice?</b><br>
A. <code>count(DISTINCT visitor_id)</code><br>
B. <code>approx_count_distinct(visitor_id)</code><br>
C. <code>collect_set(visitor_id)</code> then <code>size</code><br>
D. <code>count(visitor_id)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Approximate distinct counting is far cheaper on large data. C is exact but memory-hungry, and D counts non-null values, not unique ones.</details><br><br>

<b>Q2. Which query returns totals per (region, product), per region, and a grand total, but NOT per product alone?</b><br>
A. <code>GROUP BY CUBE (region, product)</code><br>
B. <code>GROUP BY ROLLUP (region, product)</code><br>
C. <code>GROUP BY region, product</code><br>
D. <code>GROUP BY GROUPING SETS ((region), (product))</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> ROLLUP is hierarchical. CUBE would add the per-product totals.</details><br><br>

<b>Q3. Which SQL aggregate returns the number of rows where a condition is true?</b><br>
A. <code>count_if(condition)</code><br>
B. <code>sum_if(condition)</code><br>
C. <code>count(condition)</code><br>
D. <code>filter_count(condition)</code><br>
<details><summary><b>Show answer</b></summary><b>A.</b> <code>count_if</code> is a built-in Spark SQL aggregate. <code>count(condition)</code> counts non-null results, which includes false.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>sales</code>:
<ol><li>Per region: revenue, number of orders, unique customers, and the most expensive product sold (<code>max_by</code>)</li><li>Per product: total revenue and the list of stores that sold it (unique, sorted)</li><li>Pivot: rows are regions, columns are products <code>Laptop</code>, <code>Phone</code>, <code>Monitor</code>, values are revenue with 0 instead of null</li><li>Using <code>rollup</code>, produce revenue by region and channel with region subtotals and a grand total, labelling subtotal rows as <code>TOTAL</code></li><li>Compare <code>countDistinct</code> and <code>approx_count_distinct</code> on <code>spark.range(1_000_000)</code> with <code>id % 12345</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Region KPIs
sales.groupBy("region").agg(
    F.sum("amount").alias("revenue"),
    F.count("*").alias("orders"),
    F.countDistinct("customer_id").alias("customers"),
    F.max_by("product", "amount").alias("top_product"),
).show()

# 2. Unique, sorted list of stores per product
sales.groupBy("product").agg(
    F.sum("amount").alias("revenue"),
    F.array_sort(F.collect_set("store")).alias("stores"),
).show(truncate=False)

# 3. Pivot with explicit values and zeros
sales.groupBy("region").pivot("product", ["Laptop", "Phone", "Monitor"]).sum("amount").fillna(0).show()

# 4. Rollup with labels
(
    sales.rollup("region", "channel")
    .agg(F.sum("amount").alias("revenue"), F.grouping("region").alias("g_r"), F.grouping("channel").alias("g_c"))
    .withColumn("region", F.when(F.col("g_r") == 1, "TOTAL").otherwise(F.col("region")))
    .withColumn("channel", F.when(F.col("g_c") == 1, "TOTAL").otherwise(F.col("channel")))
    .drop("g_r", "g_c")
    .orderBy("region", "channel")
    .show()
)

# 5. Exact vs approximate
spark.range(1_000_000).withColumn("k", F.col("id") % 12345).agg(
    F.countDistinct("k").alias("exact"), F.approx_count_distinct("k").alias("approx")
).show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>groupBy(...).agg(...)</code> computes many named metrics in one scan and one shuffle</li><li><code>countDistinct</code> is exact and expensive. <code>approx_count_distinct</code> is cheap with small error</li><li><code>collect_list</code> / <code>collect_set</code> gather values. Watch memory and order</li><li><code>median</code>, <code>percentile_approx</code>, <code>stddev</code> describe distributions better than <code>avg</code></li><li><code>max_by</code> / <code>min_by</code> return a value from the row with the max or min of another column</li><li><code>pivot</code> turns values into columns. Pass the values to avoid an extra job</li><li><code>rollup</code> = hierarchical subtotals, <code>cube</code> = all combinations, <code>grouping()</code> labels them</li><li><code>avg</code> is computed from partial sums and counts. Never average averages</li></ul>
</div>

| Task | Code |
|---|---|
| Many metrics | `df.groupBy("k").agg(F.sum("x").alias("s"), F.count("*").alias("n"))` |
| Distinct | `F.countDistinct("c")`, `F.approx_count_distinct("c")` |
| Collect | `F.collect_list("c")`, `F.collect_set("c")` |
| Percentile | `F.percentile_approx("x", 0.95)`, `F.median("x")` |
| Pivot | `df.groupBy("r").pivot("c", ["a", "b"]).sum("x")` |
| Subtotals | `df.rollup("a", "b")`, `df.cube("a", "b")` |
| SQL conditional count | `count_if(cond)` |

## Git commit

```
git add 01_spark_basics/12_aggregations.ipynb
git commit -m "add 12 aggregations notebook"
```